# nb11.3 - The Event-Study Plot, Twelve Ways

*Week 11, Chapter 11.3 companion. Devon comes back from the symposium with a poster that nobody
remembered. The trouble was not the design (a clean event study around major exchange listings)
but the figure: jet-colored bars, a busy gridline, no zero line, axis labels in 8-point font,
and a legend over the most important point estimate. He spent four hours on it; nobody could
read it from across the room.*

The reveal-the-trick version of the lesson: **the same numbers, plotted twelve different ways,
will tell twelve different stories - and only one or two of them pass the one-glance test from
a reader who has never seen the paper.** This notebook makes that claim concrete. We generate
a single synthetic event-study dataset (eleven leads, eleven lags, point estimates with 95% CIs)
and render it as twelve distinct figures: vertical bars vs horizontal, with vs without zero
line, color-coded by significance vs uniform, with vs without a shaded post-event region, with
vs without connector lines, in printer-safe vs full-spectrum palettes. Each variant gets a
short, honest critique. The winner gets the prize: it is the one we put in the paper.

By the end you will have an opinion about every element of an event-study figure, a Python
function `plot_es(...)` that exposes the choices as named arguments, and a comparison grid that
makes the design trade-offs visible. The figure on which Devon's next paper rests is in this
notebook.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

rng = np.random.default_rng(42)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. The synthetic event-study DGP

Twenty-three time periods relative to event time $\tau$: eleven leads ($-11, \dots, -1$), the
event period itself ($0$), and eleven lags ($1, \dots, 11$). We seed coefficients so that:

- The leads are statistically indistinguishable from zero (the parallel-trends story holds).
- The event-period coefficient and the first three lags rise from zero to a peak of about 1.5.
- The lag effects decay back toward zero by $t=+8$ (a fade-out pattern, common in finance).
- The standard errors widen at the tails (fewer observations far from the event).

These numbers are deliberately chosen to be *interesting* but *honest*: a real event study would
not look exactly like this but would have the same shape. The thing the twelve plots will fight
over is how to render this shape.

In [ ]:
def make_event_study(rng, leads=11, lags=11, peak=1.5):
    rel = np.arange(-leads, lags + 1)               # -11..11
    coefs = np.zeros_like(rel, dtype=float)
    # Pre-event: zero plus noise centered on zero.
    pre_noise = rng.normal(0, 0.08, size=leads)
    coefs[:leads] = pre_noise
    # Event period 0 onward: rise to peak by t=+2, then decay.
    post_mask = rel >= 0
    rel_post = rel[post_mask]
    base = peak * np.exp(-0.18 * np.maximum(rel_post - 2, 0)) * (1 - np.exp(-1.5 * (rel_post + 1)))
    coefs[post_mask] = base + rng.normal(0, 0.05, size=base.shape)
    # Standard errors: U-shaped, widen at the tails.
    ses = 0.15 + 0.005 * (np.abs(rel) ** 1.4)
    df = pd.DataFrame({"rel": rel, "coef": coefs, "se": ses})
    df["ci_lo"] = df["coef"] - 1.96 * df["se"]
    df["ci_hi"] = df["coef"] + 1.96 * df["se"]
    df["signif"] = ~((df["ci_lo"] < 0) & (df["ci_hi"] > 0))
    return df

es = make_event_study(rng)
es.head(10)

**Read the table.** The `rel` column is event-relative time, `coef` is the point estimate,
`se` is the standard error, and `signif` is True when the 95% CI excludes zero. Pre-event rows
(`rel < 0`) should mostly be `signif=False` - the parallel-trends pre-test is supposed to fail
to reject the null. Post-event rows should be `signif=True` for at least the first three or
four lags. If your real event study does not have this shape, you do not have an event study
yet; you have a parallel-trends violation.

## 2. The bookkeeping: a single `plot_es` function with every knob exposed

We define one workhorse plotting function. Every choice the twelve variants will toggle is a
named argument. This is the discipline: if you cannot turn the choice into an argument and pass
it from a config, you cannot reproduce the figure.

In [ ]:
def plot_es(df, ax=None, *,
            orientation="vertical",     # "vertical" or "horizontal"
            ci_style="errorbar",        # "errorbar", "ribbon", "bar"
            color_by="uniform",         # "uniform" or "signif"
            zero_line=True,
            event_band=False,
            connect_post=False,
            palette="default",          # "default", "printer", "monochrome"
            marker_size=6,
            title=""):
    if ax is None:
        fig, ax = plt.subplots()
    x = df["rel"].values; y = df["coef"].values
    lo = df["ci_lo"].values; hi = df["ci_hi"].values
    sig = df["signif"].values

    if palette == "default":
        c_pos, c_neg, c_zero = "#27ae60", "#c0392b", "#7f8c8d"
    elif palette == "printer":
        c_pos, c_neg, c_zero = "#000000", "#000000", "#666666"
    elif palette == "monochrome":
        c_pos, c_neg, c_zero = "#34495e", "#34495e", "#bdc3c7"
    else:
        raise ValueError(palette)

    if color_by == "uniform":
        colors = ["#2c3e50"] * len(x)
    elif color_by == "signif":
        colors = [c_pos if s and yi > 0 else (c_neg if s and yi < 0 else c_zero)
                  for s, yi in zip(sig, y)]
    else:
        raise ValueError(color_by)

    if event_band:
        if orientation == "vertical":
            ax.axvspan(-0.5, df["rel"].max() + 0.5, color="#ecf0f1", zorder=0)
        else:
            ax.axhspan(-0.5, df["rel"].max() + 0.5, color="#ecf0f1", zorder=0)
    if zero_line:
        (ax.axhline if orientation == "vertical" else ax.axvline)(
            0, color="black", linewidth=0.8, alpha=0.7)

    if orientation == "vertical":
        if ci_style == "errorbar":
            ax.errorbar(x, y, yerr=[y - lo, hi - y], fmt="none", ecolor=colors, capsize=3, zorder=2)
            ax.scatter(x, y, c=colors, s=marker_size ** 2, zorder=3)
        elif ci_style == "ribbon":
            ax.fill_between(x, lo, hi, color="#bdc3c7", alpha=0.5, zorder=1)
            ax.plot(x, y, marker="o", color="#2c3e50", linewidth=1.2, zorder=3)
        elif ci_style == "bar":
            for xi, yi, lo_i, hi_i, ci in zip(x, y, lo, hi, colors):
                ax.bar(xi, yi, width=0.7, color=ci, alpha=0.85, zorder=2)
                ax.plot([xi, xi], [lo_i, hi_i], color="black", linewidth=0.8, zorder=3)
        if connect_post:
            post = df[df["rel"] >= 0]
            ax.plot(post["rel"], post["coef"], color="#34495e",
                    linestyle="--", linewidth=1, zorder=2.5)
        ax.set_xlabel("event-relative time")
        ax.set_ylabel("treatment effect (coef)")
    else:
        if ci_style == "errorbar":
            ax.errorbar(y, x, xerr=[y - lo, hi - y], fmt="none", ecolor=colors, capsize=3, zorder=2)
            ax.scatter(y, x, c=colors, s=marker_size ** 2, zorder=3)
        elif ci_style == "ribbon":
            ax.fill_betweenx(x, lo, hi, color="#bdc3c7", alpha=0.5, zorder=1)
            ax.plot(y, x, marker="o", color="#2c3e50", linewidth=1.2, zorder=3)
        elif ci_style == "bar":
            for xi, yi, lo_i, hi_i, ci in zip(x, y, lo, hi, colors):
                ax.barh(xi, yi, height=0.7, color=ci, alpha=0.85, zorder=2)
                ax.plot([lo_i, hi_i], [xi, xi], color="black", linewidth=0.8, zorder=3)
        ax.set_ylabel("event-relative time")
        ax.set_xlabel("treatment effect (coef)")
    ax.set_title(title)
    return ax

# Smoke test
fig, ax = plt.subplots()
plot_es(es, ax=ax, title="smoke test")
plt.close(fig)
print("plot_es defined OK")

## 3. The twelve variants, in a four-by-three grid

We render twelve distinct event-study figures from the same data. Each variant flips one or two
of the knobs in `plot_es`. The twelve names below are intentional - they are the *vocabulary*
of event-study aesthetics, and you should be able to argue for or against each one.

In [ ]:
variants = [
    dict(title="(1) Errorbar, uniform color, zero line",
         orientation="vertical", ci_style="errorbar", color_by="uniform", zero_line=True),
    dict(title="(2) Errorbar, signif color, no zero line",
         orientation="vertical", ci_style="errorbar", color_by="signif", zero_line=False),
    dict(title="(3) Ribbon, with zero line",
         orientation="vertical", ci_style="ribbon", color_by="uniform", zero_line=True),
    dict(title="(4) Ribbon, post-event shaded band",
         orientation="vertical", ci_style="ribbon", color_by="uniform",
         zero_line=True, event_band=True),
    dict(title="(5) Bars, signif color, post-event band",
         orientation="vertical", ci_style="bar", color_by="signif",
         zero_line=True, event_band=True),
    dict(title="(6) Bars, uniform, no zero line",
         orientation="vertical", ci_style="bar", color_by="uniform", zero_line=False),
    dict(title="(7) Errorbar + connector lines on post",
         orientation="vertical", ci_style="errorbar", color_by="uniform",
         zero_line=True, connect_post=True),
    dict(title="(8) Horizontal errorbar (Cleveland)",
         orientation="horizontal", ci_style="errorbar", color_by="uniform", zero_line=True),
    dict(title="(9) Horizontal bars, signif color",
         orientation="horizontal", ci_style="bar", color_by="signif", zero_line=True),
    dict(title="(10) Printer-safe palette (black-on-white)",
         orientation="vertical", ci_style="errorbar", color_by="signif",
         zero_line=True, palette="printer"),
    dict(title="(11) Monochrome ribbon",
         orientation="vertical", ci_style="ribbon", color_by="uniform",
         zero_line=True, palette="monochrome"),
    dict(title="(12) The Devon poster (busy, no zero)",
         orientation="vertical", ci_style="bar", color_by="signif",
         zero_line=False, event_band=True, connect_post=True),
]

fig, axes = plt.subplots(4, 3, figsize=(15, 14))
for ax, spec in zip(axes.ravel(), variants):
    plot_es(es, ax=ax, **spec)
    ax.tick_params(labelsize=8)
plt.tight_layout(); plt.close(fig)
print(f"Rendered {len(variants)} variants on a 4x3 grid")

**A short critique of each.** This is the heart of the notebook - the *naming* of choices and
the *defense* of one over another.

- **(1) Errorbar, uniform color, zero line.** The Tufte-default. Quiet, readable, the eye lands
  on the zero line and then on the dots above it. Strong contender for the paper.
- **(2) Same as (1), but colored by significance.** Adds information without adding ink. Risks
  green/red conflation for color-blind readers. Acceptable if accompanied by a printer-safe
  variant.
- **(3) Ribbon with zero line.** The ribbon CI is hard to read when CIs are narrow at the
  center and wide at the tails - it can suggest more uncertainty than the errorbars do.
  Acceptable but not preferred.
- **(4) Ribbon + post-event band.** The shaded band reinforces "after this point, something
  happens." Slight aesthetic improvement over (3) but only if the band is subtle.
- **(5) Bars, signif color, post-event band.** Bars over-encode area: a coefficient of $1.5$
  vs $0.5$ looks three times bigger by area, which is misleading. Avoid bars in event studies.
- **(6) Bars, uniform, no zero line.** All the problems of (5) plus the user has to find zero.
  Reject.
- **(7) Errorbar with connector line on post.** The connector implies the post-event
  coefficients are part of a *trajectory* - which they are, conceptually. Subtle improvement
  over (1) when the post-event pattern is smooth.
- **(8) Horizontal Cleveland.** Some readers find horizontal lays easier to compare. Less
  common in finance; more common in epidemiology. Defensible.
- **(9) Horizontal bars, signif color.** Inherits the area-encoding problem from (5).
- **(10) Printer-safe palette.** This is the *always-produce-a-second-version* discipline. The
  paper PDF should be readable when printed on a black-and-white office printer.
- **(11) Monochrome ribbon.** A printer-safe ribbon alternative.
- **(12) The Devon poster.** Bars + signif color + post-event band + connector lines + no zero
  line = the figure nobody can read. This is the antimodel.

Two clear winners: **(1) for the main paper** and **(10) for the printer-safe appendix**.

## 4. The one-glance test on a single variant

A figure passes the one-glance test if a reader who has never seen the paper can answer three
questions in under fifteen seconds: (a) is the pre-event period flat at zero, (b) is the
post-event coefficient positive or negative and how big, and (c) where does the effect peak
and how fast does it fade? We will produce the winning variant at submission size and confirm
the three answers can be read off it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_es(es, ax=ax,
        orientation="vertical", ci_style="errorbar",
        color_by="uniform", zero_line=True,
        title="Headline event study: effect of major exchange listing on peer trading volume")
ax.set_xticks(np.arange(-11, 12, 2))
ax.set_ylim(-0.7, 2.1)
ax.set_xlabel("weeks relative to listing event")
ax.set_ylabel("effect on log(peer trading volume)")
plt.tight_layout(); plt.close(fig)
print("submission figure rendered (variant 1, with sharper axis labels)")

## 5. Quantifying readability: how much *ink* does each variant use?

A short Tufte-flavored diagnostic. The ink-to-data ratio is hard to compute exactly without
counting pixels, but we can proxy it: count the number of plotted elements (lines, patches,
markers) each variant places on the axes. Lower is usually better; high values flag visual
clutter.

In [ ]:
def count_artists(spec, df):
    fig, ax = plt.subplots()
    plot_es(df, ax=ax, **spec)
    n_lines = len(ax.get_lines())
    n_patches = len(ax.patches)
    n_collections = len(ax.collections)
    plt.close(fig)
    return n_lines + n_patches + n_collections

counts = []
for v in variants:
    title = v["title"]
    spec = {k: v[k] for k in v if k != "title"}
    counts.append((title, count_artists(spec, es)))
ink_df = pd.DataFrame(counts, columns=["variant", "artist_count"])
ink_df = ink_df.sort_values("artist_count").reset_index(drop=True)
print(ink_df)

**Reading the ink table.** Variants 12 (the Devon poster), 5, and 9 sit at the top of the
clutter list because they pile bars, error caps, post-event bands, and connector lines onto
the same axes. Variants 1, 3, 8, 11 sit at the bottom - they show one CI representation, the
zero line, and the data. The clean variants are the ones that pass the one-glance test.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(ink_df["variant"], ink_df["artist_count"], color="#2c3e50")
ax.set_xlabel("number of plotted elements (proxy for ink)")
ax.set_title("More ink, less signal: a Tufte-ish diagnostic of the twelve variants")
plt.tight_layout(); plt.close(fig)
ink_df

## 6. A power-of-pre-trends story the figure should tell

A reader looking at the event study wants to see two things in the leads: zero on average, and
no monotone drift. We will compute two diagnostics on the leads alone and report them next to
the figure: the mean lead coefficient and a joint F-test (here approximated by a Wald-style
$\chi^2$ on the lead coefficients as if they were independent, with an SE-weighted average).

In [ ]:
leads = es[es["rel"] < 0].copy()
mean_lead = (leads["coef"] / (leads["se"] ** 2)).sum() / ((1 / leads["se"] ** 2)).sum()
chi2 = ((leads["coef"] / leads["se"]) ** 2).sum()
print(f"Lead coefficients: mean (precision-weighted) = {mean_lead:0.4f}")
print(f"Approx Wald chi^2 on leads (assumes independence) = {chi2:0.2f}, k = {len(leads)}")

**A small caveat.** The Wald approximation here treats the lead coefficients as independent,
which they are not in any real event study (event-time leads share observations). The honest
move in your paper is to compute the joint test from the variance-covariance matrix of the
coefficient vector - `linearmodels` and `pyfixest` both give it back. This notebook is teaching
the figure-discipline part of the story; for the formal pre-test, see the chapter on
event studies in Week 9.

## 7. Producing the camera-ready figure with print-publication settings

Submission to JF, RFS, JFE wants vector output (PDF), 9-point sans serif for axis labels,
and a defined linewidth. We will set these and re-export the winner.

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.linewidth": 0.8,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.1,
    "lines.markersize": 5,
    "figure.dpi": 110,
})
fig, ax = plt.subplots(figsize=(7.0, 4.0))   # JF-style two-column width
plot_es(es, ax=ax, orientation="vertical", ci_style="errorbar",
        color_by="uniform", zero_line=True, title="")
ax.set_xticks(np.arange(-11, 12, 2))
ax.set_ylim(-0.7, 2.1)
ax.set_xlabel("weeks relative to listing event")
ax.set_ylabel("effect on log(peer trading volume)")
plt.tight_layout(); plt.close(fig)
print("camera-ready figure rendered at JF two-column width")

## 8. When the twelve-way exercise itself fails (be honest)

Two failure modes you should anticipate.

1. **Overfitting on one journal's style.** The conventions of *Journal of Finance* are not the
   conventions of *American Economic Review* or *Nature*. The eleven you reject for your JF
   submission might be exactly the figure *Nature* wants. The lesson generalizes; the choice
   of winner does not.

2. **One ugly figure, one beautiful conclusion.** A figure can pass the one-glance test and
   still be misleading - if the y-axis is truncated, the zero line is missing, or the CI is
   computed with a stale standard error. Aesthetics and statistical honesty are independent
   axes; you need both. Re-running the diagnostic in section 6 every time you re-export the
   figure is the cheapest safeguard.

## 9. Your turn

1. **Add a thirteenth variant.** Devise one knob the twelve do not yet vary (for instance: log
   y-axis vs linear), implement it as a `plot_es` argument, and add the variant to the grid.
   Argue for or against it.

2. **Color-blind audit.** Take variant (2) - the signif-colored errorbar - and run it through
   a color-blind simulator (Python: `daltonlens`; or the free Coblis web tool). Decide whether
   the green/red distinction survives. If it does not, propose a fix.

3. **Time-series sanity.** Re-run `make_event_study` with `peak=0.0`. Confirm that all twelve
   variants visually show "no effect" without misleading the reader. Variants that *seem* to
   show an effect on null data are flagged for redesign.

4. **The poster vs the paper.** Pick a winner for the conference poster and a (possibly
   different) winner for the paper. Defend the choice in two sentences each. The right answer
   may differ: a poster is read from twelve feet away; the paper is read at arm's length.

**Citations to anchor your writeup.** Tufte (2001, *The Visual Display of Quantitative
Information*, 2nd ed.); Cleveland (1985, *The Elements of Graphing Data*); Wickham (2016,
*ggplot2*, ch. 4-6) on visual encoding; Callaway and Sant'Anna (2021, *J Econometrics*
225(2):200-230) on the event-study estimator the figure is rendering.